# Homer Transformer

Un Transformer **encodeur-décodeur** codé from scratch :

1. **L'encodeur** lit la phrase de l'utilisateur (il voit toute la phrase d'un coup).
2. **Le décodeur** écrit la réponse façon Homer, un token à la fois. Il ne voit que ce qu'il a déjà écrit
   (masque causal) et va chercher l'information dans la sortie de l'encodeur (cross-attention).

Le dataset : les dialogues (en français) de 473 épisodes des Simpsons. On apprend au modèle à répondre
comme Homer à la réplique d'un autre personnage, par exemple `Marge: Tu as fini ?` → `Homer: D'oh !`.

Plan du notebook : briques du modèle → tokenizer BPE (tiktoken) → données → entraînement → discussion avec Homer.

## Installation (Google Colab)

Sur Colab, avant de lancer quoi que ce soit : **Exécution → Modifier le type d'exécution → GPU T4**.

La cellule suivante installe `tiktoken` et télécharge le dataset depuis Hugging Face s'il n'est pas déjà là.
Sur ton PC, elle ne fait rien de plus que vérifier que le dataset est présent.

In [ ]:
import sys
import urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules   # True si on tourne sur Google Colab

if IN_COLAB:
    !pip install -q tiktoken

dataset_file = Path("data/Simpsons/dialogues.jsonl")
if not dataset_file.exists():
    dataset_file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://huggingface.co/datasets/Romaindcrl/Simpsons/resolve/main/dialogues.jsonl"
    urllib.request.urlretrieve(url, dataset_file)
print("dataset :", dataset_file, f"({dataset_file.stat().st_size / 1e6:.1f} Mo)")

In [35]:
import json
import random
import time
from pathlib import Path

import tiktoken
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Head(nn.Module):
    """Une tête d'attention : self-attention si context est None, cross-attention sinon."""

    def __init__(self, n_embd: int, head_size: int, dropout: float) -> None:
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, context=None):
       
        source = x if context is None else context

        q = self.query(x)          # les questions viennent toujours de x (celui qui lit)
        k = self.key(source)       # les étiquettes viennent de la source
        v = self.value(source)     # les informations aussi

        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T_x, T_source)
        if mask is not None:
            wei = wei.masked_fill(mask == 0, float("-inf"))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ v                                         # (B, T_x, head_size)


class MultiHeadAttention(nn.Module):
    """Plusieurs têtes en parallèle, concaténées puis projetées."""

    def __init__(self, n_embd: int, n_head: int, dropout: float) -> None:
        super().__init__()
        self.heads = nn.ModuleList(Head(n_embd, n_embd // n_head, dropout) for _ in range(n_head))
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, context=None):
        return self.dropout(self.proj(torch.cat([h(x, mask, context) for h in self.heads], dim=-1)))


class FeedForward(nn.Module):
    """MLP appliqué à chaque position indépendamment."""

    def __init__(self, n_embd: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    """Bloc Transformer pre-norm : attention puis feed-forward, avec résiduels."""

    def __init__(self, n_embd: int, n_head: int, dropout: float) -> None:
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        x = x + self.sa(self.ln1(x), mask)
        return x + self.ffwd(self.ln2(x))


class GPTLanguageModel(nn.Module):
    """Transformer générique : décodeur avec un masque causal, encodeur sans masque."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(Block(n_embd, n_head, dropout) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        """Renvoie les logits `(B, T, vocab_size)`."""
        _, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x, mask)
        return self.lm_head(self.ln_f(x))



In [37]:
def causal_mask(T: int, device=None) -> torch.Tensor:
    """(T, T) : chaque position ne voit qu'elle-même et le passé (décodeur)."""
    return torch.tril(torch.ones(T, T, device=device, dtype=torch.bool))


def padding_mask(pad: torch.Tensor) -> torch.Tensor:
    """pad : (B, T) avec 1 = vrai caractère, 0 = remplissage -> (B, 1, T) : personne ne regarde le remplissage."""
    return pad.bool()[:, None, :]
class Encoder(nn.Module):
    """Chaque position voit toute la séquence. Renvoie un vecteur par position."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(Block(n_embd, n_head, dropout) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(n_embd)
        # pas de lm_head : l'encodeur ne prédit rien, il produit des représentations

    def forward(self, idx: torch.Tensor, pad: torch.Tensor | None = None) -> torch.Tensor:
        """idx : (B, T). pad optionnel : (B, T), 1 = vrai caractère, 0 = remplissage.
        Renvoie `(B, T, n_embd)`."""
        _, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))

        # pas de masque causal : seulement le padding, s'il y en a
        mask = padding_mask(pad) if pad is not None else None

        for block in self.blocks:
            x = block(x, mask)
        return self.ln_f(x)




In [38]:
class DecoderBlock(nn.Module):
    """Bloc décodeur pre-norm : self-attention masquée, cross-attention, puis feed-forward."""

    def __init__(self, n_embd: int, n_head: int, dropout: float) -> None:
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, dropout)    # self-attention
        self.ca = MultiHeadAttention(n_embd, n_head, dropout)    # cross-attention
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, enc_out, self_mask=None, cross_mask=None):
        # 1. le texte du décodeur se regarde lui-même (sans voir le futur)
        x = x + self.sa(self.ln1(x), self_mask)

        # 2. le décodeur va chercher l'information dans la sortie de l'encodeur
        x = x + self.ca(self.ln2(x), cross_mask, context=enc_out)

        # 3. chaque position réfléchit seule
        return x + self.ffwd(self.ln3(x))

In [39]:
mha = MultiHeadAttention(n_embd=32, n_head=4, dropout=0.0)
x = torch.randn(2, 5, 32)
enc = torch.randn(2, 9, 32)
print(mha(x).shape)               # torch.Size([2, 5, 32])
print(mha(x, context=enc).shape)

torch.Size([2, 5, 32])
torch.Size([2, 5, 32])


## Le décodeur complet et le modèle encodeur-décodeur

On empile les `DecoderBlock`, comme l'`Encoder` empile les `Block`. Le décodeur a en plus une `lm_head`
qui transforme chaque vecteur en un score pour chaque mot du vocabulaire (les *logits*).

In [40]:
class Decoder(nn.Module):
    """Écrit la réponse. Chaque position ne voit que le passé, et regarde la sortie de l'encodeur."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(DecoderBlock(n_embd, n_head, dropout) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)   # un score par token du vocabulaire

    def forward(self, idx, enc_out, enc_pad=None):
        """idx : (B, T) la réponse en cours. enc_out : (B, T_src, n_embd). enc_pad : (B, T_src).
        Renvoie les logits `(B, T, vocab_size)`."""
        _, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))

        # self-attention : masque causal (pas le droit de regarder le futur).
        # Pas besoin de masquer le remplissage ici : il est toujours À LA FIN, donc dans le futur
        # des vrais tokens, et le masque causal le cache déjà.
        self_mask = causal_mask(T, idx.device)
        # cross-attention : on ne regarde pas le remplissage de la question
        cross_mask = padding_mask(enc_pad) if enc_pad is not None else None

        for block in self.blocks:
            x = block(x, enc_out, self_mask, cross_mask)
        return self.lm_head(self.ln_f(x))


class HomerTransformer(nn.Module):
    """Encodeur (lit la question) + décodeur (écrit la réponse)."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.block_size = block_size
        self.encoder = Encoder(vocab_size, block_size, n_embd, n_head, n_layer, dropout)
        self.decoder = Decoder(vocab_size, block_size, n_embd, n_head, n_layer, dropout)

    def forward(self, src, src_pad, tgt_in):
        enc_out = self.encoder(src, src_pad)              # (B, T_src, n_embd)
        return self.decoder(tgt_in, enc_out, src_pad)     # (B, T_tgt, vocab_size)

## Hyperparamètres

Tout ce qu'on peut régler est ici. Si l'entraînement est trop lent sur CPU, baisse `N_EMBD`, `N_LAYER`
ou `MAX_LEN`.

In [ ]:
# --- données ---
DATA_PATH = Path("data/Simpsons/dialogues.jsonl")   # les dialogues des épisodes
MAX_LEN = 64                         # nombre max de tokens pour la question ET pour la réponse
VAL_RATIO = 0.1                      # 10 % des exemples servent à la validation

# --- modèle ---
N_EMBD = 512        # taille des vecteurs
N_HEAD = 4          # nombre de têtes d'attention (N_EMBD doit être divisible par N_HEAD)
N_LAYER = 4         # nombre de blocs dans l'encodeur et dans le décodeur
DROPOUT = 0.1

# --- entraînement ---
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 3e-4
CHECKPOINT_PATH = Path("homer_transformer.pt")

device = "cuda" if torch.cuda.is_available() else "cpu"   # GPU si disponible
torch.manual_seed(42)
random.seed(42)
print("device :", device)

## Tokenizer BPE (tiktoken)

On utilise le tokenizer BPE de GPT-2 : il découpe le texte en morceaux de mots fréquents
(`"donuts"` → `"don"` + `"uts"`). C'est bien plus précis qu'un découpage caractère par caractère.
Il a été appris sur de l'anglais : les mots français sont coupés en un peu plus de morceaux, mais ça marche.

Il connaît 50 257 tokens. On en ajoute deux à nous :
- `PAD` : le remplissage, pour mettre toutes les phrases d'un batch à la même longueur ;
- `BOS` : « début de la réponse », le premier token que reçoit le décodeur.

Pour la fin de la réponse (`EOS`), on réutilise le token `<|endoftext|>` de GPT-2.

In [42]:
tokenizer = tiktoken.get_encoding("gpt2")

EOS_ID = tokenizer.eot_token        # 50256 : <|endoftext|>  -> "la réponse est finie"
PAD_ID = tokenizer.n_vocab          # 50257 : nouveau token  -> remplissage
BOS_ID = tokenizer.n_vocab + 1      # 50258 : nouveau token  -> "commence la réponse"
VOCAB_SIZE = tokenizer.n_vocab + 2  # 50259 tokens au total


def encode(text: str) -> list[int]:
    return tokenizer.encode_ordinary(text)


def decode(ids: list[int]) -> str:
    # on retire nos tokens spéciaux, que tiktoken ne connaît pas
    return tokenizer.decode([i for i in ids if i < EOS_ID])


ids = encode("D'oh! Mmm... donuts")
print(ids)
print([decode([i]) for i in ids])   # on voit le découpage en morceaux
print(decode(ids))

[35, 6, 1219, 0, 337, 3020, 986, 836, 5500]
['D', "'", 'oh', '!', ' M', 'mm', '...', ' don', 'uts']
D'oh! Mmm... donuts


## Les données

Le fichier `dialogues.jsonl` contient **un épisode par ligne**, au format JSON :

```
{"title": "Les secrets d'un mariage réussi", "text": "Marge: Oh, Homer...\nHomer: Prendre un bain au whisky ?\n..."}
```

Dans `text`, chaque ligne est une réplique `Personnage: phrase`, et une ligne vide marque un changement de scène.

On fabrique une paire **(réplique, réponse d'Homer)** à chaque fois qu'Homer répond à quelqu'un dans la même scène :

```
entrée  : "Marge: Il y a ça. Et il y a aussi les cours d'éducation pour adultes."
réponse : "Oh ! Et en quoi l'é-du-ca-tion peut me rendre plus intelligent ? ..."
```

On garde le nom du personnage dans l'entrée : le modèle apprend qui lui parle, et on pourra lui écrire
`Marge: ...` pour discuter avec lui.

In [43]:
def load_pairs(path: Path) -> list[tuple[str, str]]:
    """Renvoie la liste des paires (réplique d'un personnage, réponse d'Homer)."""
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:                                  # une ligne = un épisode
            episode = json.loads(line)
            previous = None                             # la réplique d'avant, dans la même scène

            for replique in episode["text"].split("\n"):
                if ": " not in replique:                # ligne vide = nouvelle scène
                    previous = None
                    continue

                speaker, words = replique.split(": ", 1)    # "Marge: Bonjour" -> "Marge", "Bonjour"
                speaker, words = speaker.strip(), words.strip()

                # quelqu'un d'autre parle, puis Homer répond -> une paire
                if speaker == "Homer" and words and previous and not previous.startswith("Homer:"):
                    pairs.append((previous, words))

                previous = f"{speaker}: {words}"
    return pairs


pairs = load_pairs(DATA_PATH)
print(f"{len(pairs)} paires chargées depuis {DATA_PATH}")
print(f"dont {sum(q.startswith('Marge:') for q, _ in pairs)} où Homer répond à Marge")
for question, answer in pairs[:3]:
    print(question, "  ->  Homer:", answer)

17720 paires chargées depuis data\Simpsons\dialogues.jsonl
dont 4389 où Homer répond à Marge
Bart: Ensuite, il a prétendu que c'était lui qui avait monté les chats et les chiens les uns contre les autres ! Pourquoi il invente toujours des histoires débiles ?   ->  Homer: Il est pit-être temps de mettre grand-père dans une maison.
Lisa: Non, les gens âgés méritent notre respect. Regarde Jacques Cousteau et Goldie Hawn, tu les enfermerais pas comme des citoyens de second ordre !   ->  Homer: Pfff ! De second ordre, ho, ho ! Et la location vieillesse, les tickets de bus gratos, les aides à domicile, les lunettes à double foyer, les futals qui remontent jusque sous les bras et tous les autres avantages ! Hou, tu veux que je te dise, les vieux ils ont la belle vie !
Abraham: Eh ! Ouvrez-moi ! Quelqu'un essaie de me tuer ! Ooooh, Jésus Marie Joseph ! Faites qu'ils m'ouvrent la porte !!   ->  Homer: Qui qu'est là ?


### Dataset et DataLoader

Pour chaque paire on fabrique :
- `src` : la question tokenisée (pour l'encodeur) ;
- `tgt` : `[BOS] + réponse + [EOS]` (pour le décodeur).

Pendant l'entraînement, le décodeur reçoit `tgt[:-1]` et doit prédire `tgt[1:]` : à chaque position,
il doit deviner le token **suivant**.

```
entrée du décodeur : [BOS]  Woo   -hoo   !
ce qu'il doit dire : Woo    -hoo  !      [EOS]
```

Les phrases n'ont pas toutes la même longueur : la fonction `collate` les complète avec des `PAD`
pour former un tableau rectangulaire.

In [44]:
class HomerDataset(Dataset):
    def __init__(self, pairs: list[tuple[str, str]]) -> None:
        self.examples = []
        for question, answer in pairs:
            src = encode(question)[:MAX_LEN]
            tgt = [BOS_ID] + encode(answer)[:MAX_LEN - 2] + [EOS_ID]   # -2 : place pour BOS et EOS
            self.examples.append((torch.tensor(src), torch.tensor(tgt)))

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


def collate(batch):
    """Met toutes les séquences du batch à la même longueur en ajoutant des PAD à la fin."""
    srcs, tgts = zip(*batch)
    src = pad_sequence(srcs, batch_first=True, padding_value=PAD_ID)   # (B, T_src)
    tgt = pad_sequence(tgts, batch_first=True, padding_value=PAD_ID)   # (B, T_tgt)
    src_pad = (src != PAD_ID).long()                                   # 1 = vrai token, 0 = remplissage
    return src, src_pad, tgt


# séparation entraînement / validation
random.shuffle(pairs)
n_val = max(1, int(len(pairs) * VAL_RATIO))
train_ds = HomerDataset(pairs[n_val:])
val_ds = HomerDataset(pairs[:n_val])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

print(f"train : {len(train_ds)} exemples, val : {len(val_ds)} exemples")
src, src_pad, tgt = next(iter(train_loader))
print("src", tuple(src.shape), "| tgt", tuple(tgt.shape))

train : 15948 exemples, val : 1772 exemples
src (32, 64) | tgt (32, 64)


## Création du modèle

`AdamW` est l'optimiseur classique pour les Transformers : il met à jour les poids à partir des gradients.

In [45]:
# les réglages du modèle, gardés dans un dictionnaire pour les sauvegarder avec les poids
model_config = dict(
    vocab_size=VOCAB_SIZE, block_size=MAX_LEN, n_embd=N_EMBD,
    n_head=N_HEAD, n_layer=N_LAYER, dropout=DROPOUT,
)
model = HomerTransformer(**model_config).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params / 1e6:.1f} M paramètres")

106.7 M paramètres


## Les fonctions d'entraînement

- `compute_loss` : passe un batch dans le modèle et calcule l'erreur (cross-entropy). Les positions `PAD`
  sont ignorées (`ignore_index`) : on ne pénalise pas le modèle sur le remplissage.
- `train_one_epoch` : un passage complet sur les données d'entraînement, en mettant à jour les poids.
- `evaluate` : même calcul sur la validation, **sans** toucher aux poids. Si la loss de validation
  remonte alors que celle d'entraînement baisse, le modèle apprend par cœur (overfitting).

In [46]:
def compute_loss(model, batch):
    src, src_pad, tgt = (t.to(device) for t in batch)
    tgt_in = tgt[:, :-1]     # ce que le décodeur reçoit
    tgt_out = tgt[:, 1:]     # ce qu'il doit prédire (décalé d'un cran)

    logits = model(src, src_pad, tgt_in)                  # (B, T, VOCAB_SIZE)
    return F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),                   # (B*T, VOCAB_SIZE)
        tgt_out.reshape(-1),                              # (B*T,)
        ignore_index=PAD_ID,
    )


def train_one_epoch(model, loader, optimizer) -> float:
    model.train()                   # active le dropout
    total = 0.0
    for i, batch in enumerate(loader, 1):
        loss = compute_loss(model, batch)
        optimizer.zero_grad()       # 1. on efface les anciens gradients
        loss.backward()             # 2. on calcule les nouveaux
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # évite les gradients qui explosent
        optimizer.step()            # 3. on met à jour les poids
        total += loss.item()
        if i % 100 == 0:            # pour voir que ça avance
            print(f"   batch {i}/{len(loader)} | loss {total / i:.3f}")
    return total / len(loader)


@torch.no_grad()                    # pas de gradients : plus rapide et moins de mémoire
def evaluate(model, loader) -> float:
    model.eval()                    # désactive le dropout
    total = 0.0
    for batch in loader:
        total += compute_loss(model, batch).item()
    return total / len(loader)

## Générer une réponse

Le décodeur écrit un token à la fois :
1. on encode la question une seule fois ;
2. on part de `[BOS]`, on demande au décodeur le token suivant, on l'ajoute, et on recommence ;
3. on s'arrête quand il produit `EOS` (ou quand la réponse est trop longue).

Pour choisir le token suivant :
- `temperature` : < 1 = réponses plus sûres, > 1 = plus folles ;
- `top_k` : on tire au sort seulement parmi les `k` tokens les plus probables.

In [47]:
@torch.no_grad()
def reply(model, question: str, max_new_tokens: int = 50, temperature: float = 0.8, top_k: int = 50) -> str:
    model.eval()
    src = torch.tensor([encode(question)[:MAX_LEN]], device=device)   # (1, T_src)
    src_pad = torch.ones_like(src)                                    # pas de remplissage : une seule phrase
    enc_out = model.encoder(src, src_pad)                             # on lit la question une seule fois

    out = [BOS_ID]
    for _ in range(min(max_new_tokens, MAX_LEN - 1)):
        idx = torch.tensor([out], device=device)                     # (1, T) la réponse jusqu'ici
        logits = model.decoder(idx, enc_out, src_pad)[0, -1]         # scores du token suivant : (VOCAB_SIZE,)
        logits[PAD_ID] = logits[BOS_ID] = float("-inf")              # il ne doit jamais écrire ces deux-là

        logits = logits / temperature
        kth_best = torch.topk(logits, top_k).values[-1]
        logits[logits < kth_best] = float("-inf")                    # on garde seulement les top_k
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()     # tirage au sort selon les probas

        if next_id == EOS_ID:
            break
        out.append(next_id)
    return decode(out[1:])


print(reply(model, "Marge: Homer, tu as fini ton travail ?"))   # avant entraînement : n'importe quoi

 Leonardo expansivepants smokes Flor Augusta Commander characterundle driftingunders Bangalore Employees numbered EN Mutual Number Age strategies allowasha NJ ** greatness wastedstarting TL safest tarJean guilt FullzymortRELGoogle cracks faced joystick unique filler challengingiddles potatoes reperc testimonies Repeat syrup consideredSony


## La boucle d'entraînement

À chaque epoch : on entraîne, on évalue, et on garde le meilleur modèle (celui qui a la plus petite
loss de validation). On affiche aussi une réponse d'exemple pour voir Homer progresser.

In [48]:
TEST_QUESTION = "Marge: Homer, tu as fini ton travail ?"
best_val = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_loss = evaluate(model, val_loader)
    history.append((train_loss, val_loss))

    saved = ""
    if val_loss < best_val:
        best_val = val_loss
        # on sauvegarde les poids ET les réglages, pour pouvoir reconstruire le même modèle
        torch.save({"config": model_config, "weights": model.state_dict()}, CHECKPOINT_PATH)
        saved = " (sauvegardé)"

    print(f"epoch {epoch:3d} | train {train_loss:.3f} | val {val_loss:.3f} | {time.time() - start:.1f}s{saved}")
    if epoch % 5 == 0 or epoch == EPOCHS:
        print("   Homer :", reply(model, TEST_QUESTION))

KeyboardInterrupt: 

## Récupérer le modèle entraîné (Colab)

Quand Colab se déconnecte, tous ses fichiers sont effacés. Cette cellule télécharge `homer_transformer.pt`
sur ton ordinateur : mets-le ensuite dans le dossier du projet pour discuter avec Homer en local.

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(CHECKPOINT_PATH))

## Discuter avec Homer

On recharge le meilleur modèle sauvegardé (il faut avoir fini au moins une epoch d'entraînement),
puis on lui parle. Le modèle est reconstruit avec les réglages enregistrés dans la sauvegarde :
ça marche même si tu as changé `N_EMBD` depuis. Tu joues Marge : écris juste ta phrase, on ajoute `Marge: ` devant.
Tape `quit` (ou rien) pour arrêter.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model = HomerTransformer(**checkpoint["config"]).to(device)   # même taille que le modèle sauvegardé
model.load_state_dict(checkpoint["weights"])

for question in ["Marge: Homer, tu veux une salade ?", "Marge: Homer, la centrale a appelé.", "Marge: Tu m'aimes ?"]:
    print(question)
    print("Homer:", reply(model, question))
    print()

In [ ]:
while True:
    question = input("Marge: ")
    if question.strip().lower() in ("", "quit"):
        break
    print("Marge:", question)
    print("Homer:", reply(model, f"Marge: {question}"))